# Apo HPV16 E6: auditable two-state deeptime MSM

This notebook rebuilds the E6HL-contact MSM as one coherent analysis. The same
base Bayesian MSM (and its maximum-likelihood prior) is used for the reported
stationary populations, two-state transition matrix, model-extrapolated MFPTs,
and Chapman–Kolmogorov (CK) predictions.

Key safeguards:

- physical time is defined once and checked against the XTC timestamps;
- replicas remain separate during TICA and transition counting;
- all integer lags and their physical-time conversions are printed;
- raw sliding counts and statistically adjusted effective counts are both
  reported, with the effective-count model declared as the inferential model;
- CK estimates are compared with predictions from the exact base MSM used for
  the kinetic summary;
- each replica's raw macrostate timeline and bidirectional transition counts are
  shown;
- point estimates and percentile-based 95% credible intervals are reported
  instead of posterior mean ± SD for the right-skewed MFPT distribution.

The CK test spans 50–250 ns, far below the slowest implied timescale. Agreement
at these lags is a limited internal-consistency check; it does **not** establish
well-converged equilibrium populations or tens-of-microseconds kinetics.


In [ ]:
"""Fit the E6HL TICA/MSM and compute populations, MFPTs and CK diagnostics.
Inputs: the combined 35,002-frame XTC, matching topology and optional saved labels.
Outputs: MSM numerical bundle, diagnostic tables, network and CK/ITS plots.
Supports V12 Figure 5g–h and Figure S4; historical output names retain Figure4.
"""
from pathlib import Path
import csv
import json
import os
import platform
import sys

# Keep plotting caches and generated files beside this analysis.
os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mplconfig"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import deeptime
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch
import mdtraj as md
import numpy as np

from deeptime.clustering import KMeans
from deeptime.decomposition import TICA
from deeptime.markov import TransitionCountEstimator
from deeptime.markov.msm import BayesianMSM, MaximumLikelihoodMSM

# ----------------------------- declared constants -----------------------------
RANDOM_SEED = 42
# Working physical-time declaration from the stated 10 + 10 + 5 + 10 µs
# replica durations and corresponding frame counts. The XTC timestamp audit
# below is mandatory because the current combined file contains mixed metadata.
NS_PER_FRAME = 1.0
TICA_LAG_FRAMES = 10
TICA_DIM = 4
N_MICROSTATES = 6
MSM_LAG_FRAMES = 50
MSM_COUNT_MODE = "effective"
N_POSTERIOR_SAMPLES = 500
REPLICA_LENGTHS = (10000, 10000, 5001, 10001)
CK_MULTIPLES = (1, 2, 3, 4, 5)
ITS_LAGS_FRAMES = (1, 2, 5, 10, 20, 30, 40, 50, 60, 75, 100, 150, 200, 250)
REUSE_EXISTING_MICROSTATE_CACHE = os.environ.get("APO16E6_REUSE_MICROSTATE_CACHE") == "1"

np.random.seed(RANDOM_SEED)
np.set_printoptions(precision=6, suppress=True, linewidth=140)

# Input/output configuration. Execute with the notebook directory as working directory.
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "environment.yml").is_file() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "environment.yml").is_file():
    raise FileNotFoundError("Run this notebook from within the repository.")
TRAJECTORY = Path(os.environ.get("APO16E6_TRAJECTORY", str(REPO_ROOT / "trajectories/trajectory_35us.xtc")))
TOPOLOGY = Path(os.environ.get("APO16E6_TOPOLOGY", str(REPO_ROOT / "structures/topology_apo16e6.pdb")))
RESULTS_DIR = Path(os.environ.get("APO16E6_RESULTS_DIR", str(REPO_ROOT / "results/msm")))
DATA_DIR = TRAJECTORY.parent
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def save_figure(fig, stem):
    fig.savefig(RESULTS_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
    fig.savefig(RESULTS_DIR / f"{stem}.pdf", bbox_inches="tight")

print(f"Python       : {platform.python_version()}")
print(f"deeptime     : {deeptime.__version__}")
print(f"MDTraj       : {md.__version__}")
print(f"NumPy        : {np.__version__}")
print(f"Matplotlib   : {matplotlib.__version__}")
print(f"Data         : {DATA_DIR}")
print(f"Results      : {RESULTS_DIR.resolve()}")
print(f"Random seed  : {RANDOM_SEED}")
print(f"Reuse cache  : {REUSE_EXISTING_MICROSTATE_CACHE}")


## 1. Load the four replicas and verify physical time

The concatenated XTC is split back into the four independent trajectories before
any time-lagged operation. No transition is counted across a replica boundary.
The present combined XTC is also audited replica by replica because a single
global median can conceal mixed timestamp strides.


In [ ]:
traj = md.load(TRAJECTORY, top=TOPOLOGY)
if traj.n_frames != sum(REPLICA_LENGTHS):
    raise ValueError(f"Expected {sum(REPLICA_LENGTHS)} frames, found {traj.n_frames}")

starts = np.cumsum((0,) + REPLICA_LENGTHS[:-1])
rep_boundaries = [(int(s), int(s + n)) for s, n in zip(starts, REPLICA_LENGTHS)]

# XTC times are stored in ps. Inspect each original replica separately; the
# fourth segment resets its clock and the first three use a different increment.
detected_ns_per_frame_by_replica = []
timestamp_duration_us_by_replica = []
for s, e in rep_boundaries:
    replica_time_ps = np.asarray(traj.time[s:e], dtype=float)
    positive_dt_ps = np.diff(replica_time_ps)
    positive_dt_ps = positive_dt_ps[positive_dt_ps > 0]
    detected_ns = float(np.median(positive_dt_ps) / 1000.0) if positive_dt_ps.size else np.nan
    detected_ns_per_frame_by_replica.append(detected_ns)
    timestamp_duration_us_by_replica.append(
        float((replica_time_ps[-1] - replica_time_ps[0]) / 1_000_000.0)
    )

timestamp_matches_declaration = bool(np.allclose(
    detected_ns_per_frame_by_replica,
    NS_PER_FRAME,
    rtol=0,
    atol=1e-6,
    equal_nan=False,
))
timebase_status = (
    "XTC timestamps agree with the declared frame spacing"
    if timestamp_matches_declaration
    else "DECLARED 1 ns/frame OVERRIDES INCONSISTENT XTC TIMESTAMPS; source export must be verified"
)

print(f"Loaded        : {traj.n_frames} frames, {traj.n_atoms} atoms, {traj.n_residues} residues")
print(f"Replica sizes : {list(REPLICA_LENGTHS)} frames")
print(f"Declared frame spacing              : {NS_PER_FRAME:.6g} ns/frame")
print(f"XTC median spacing by replica (ns)  : {detected_ns_per_frame_by_replica}")
print(f"XTC timestamp span by replica (µs)  : {timestamp_duration_us_by_replica}")
print(f"TIMEBASE STATUS                     : {timebase_status}")
if not timestamp_matches_declaration:
    print("\nCRITICAL: Replicas 1–3 carry ~0.1-ns timestamp increments, whereas replica 4 "
          "carries ~1-ns increments. The 1-ns conversion below follows the declared "
          "10 + 10 + 5 + 10 µs design, not uniform metadata in the combined XTC. "
          "Do not publish physical kinetics until the source trajectories/export log "
          "confirm which timebase is correct and the XTC is rebuilt consistently.\n")
print(f"Aggregate time: {traj.n_frames * NS_PER_FRAME / 1000:.3f} µs "
      "(not one continuous trajectory)")
print(f"TICA lag      : {TICA_LAG_FRAMES:d} frames = "
      f"{TICA_LAG_FRAMES * NS_PER_FRAME:.1f} ns")
print(f"MSM lag       : {MSM_LAG_FRAMES:d} frames = "
      f"{MSM_LAG_FRAMES * NS_PER_FRAME:.1f} ns")


## 2. E6HL contact features, TICA, and six microstates

Features are all closest-heavy-atom distances between nonadjacent residues in
E6HL residues 62–79. This produces 136 features. No zinc-ligand masking is
performed here; any manuscript statement that such pairs were masked would be
inconsistent with this code and should be corrected or implemented explicitly.


In [ ]:
e6hl_residue_indices = [
    residue.index for residue in traj.topology.residues
    if 62 <= residue.resSeq <= 79
]
resseq_by_index = {residue.index: residue.resSeq for residue in traj.topology.residues}

feature_pairs = np.asarray([
    (i, j)
    for a, i in enumerate(e6hl_residue_indices)
    for j in e6hl_residue_indices[a + 1:]
    if abs(resseq_by_index[i] - resseq_by_index[j]) > 1
], dtype=int)
if feature_pairs.shape != (136, 2):
    raise ValueError(f"Expected 136 E6HL contact features, found {len(feature_pairs)}")

cache_path = RESULTS_DIR / "apo16e6_msm_numeric_bundle.npz"
if REUSE_EXISTING_MICROSTATE_CACHE and cache_path.is_file():
    cache = np.load(cache_path)
    if not np.array_equal(cache["replica_lengths"], np.asarray(REPLICA_LENGTHS)):
        raise ValueError("Cached replica lengths do not match the declared analysis")
    if int(cache["tica_lag_frames"]) != TICA_LAG_FRAMES:
        raise ValueError("Cached TICA lag does not match the declared analysis")
    if int(cache["msm_lag_frames"]) != MSM_LAG_FRAMES:
        raise ValueError("Cached MSM lag does not match the declared analysis")
    all_dtrajs = cache["discrete_trajectories_concatenated"].astype(int)
    dtrajs = [all_dtrajs[s:e] for s, e in rep_boundaries]
    print(f"Loaded explicit microstate cache: {cache_path}")
    print("Set APO16E6_REUSE_MICROSTATE_CACHE=0 or unset it for a full feature/TICA/k-means rebuild.")
else:
    features, _ = md.compute_contacts(traj, contacts=feature_pairs, scheme="closest-heavy")
    feature_trajectories = [features[s:e] for s, e in rep_boundaries]

    tica_model = TICA(lagtime=TICA_LAG_FRAMES, dim=TICA_DIM).fit_fetch(feature_trajectories)
    tica_trajectories = [tica_model.transform(x) for x in feature_trajectories]
    all_tica = np.vstack(tica_trajectories)

    cluster_model = KMeans(
        n_clusters=N_MICROSTATES,
        fixed_seed=RANDOM_SEED,
        max_iter=1000,
        n_jobs=1,
    ).fit_fetch(all_tica)
    dtrajs = [cluster_model.transform(x).astype(int) for x in tica_trajectories]
    all_dtrajs = np.concatenate(dtrajs)

    print(f"Feature matrix : {features.shape}")
    print(f"TICA output    : {all_tica.shape}")
    print(f"TICA singular values (first 5): {np.asarray(tica_model.singular_values[:5])}")
for i, dtraj in enumerate(dtrajs, start=1):
    values, counts_i = np.unique(dtraj, return_counts=True)
    print(f"Replica {i} microstate counts: {dict(zip(values.tolist(), counts_i.tolist()))}")


## 3. One declared base MSM

The inferential model uses effective counts at a 50-frame (50-ns) lag and a
reversible Bayesian MSM. Its maximum-likelihood prior is the point-estimate MSM.
Raw sliding counts are retained only as an auditable count report; they are not
silently substituted into the kinetic model.


In [ ]:
raw_count_model = (
    TransitionCountEstimator(lagtime=MSM_LAG_FRAMES, count_mode="sliding")
    .fit_fetch(dtrajs)
    .submodel_largest()
)
effective_count_model = (
    TransitionCountEstimator(lagtime=MSM_LAG_FRAMES, count_mode=MSM_COUNT_MODE)
    .fit_fetch(dtrajs)
    .submodel_largest()
)

if effective_count_model.n_states != N_MICROSTATES:
    raise ValueError(
        f"Largest connected set contains {effective_count_model.n_states}/{N_MICROSTATES} microstates"
    )
if not np.array_equal(raw_count_model.state_symbols, effective_count_model.state_symbols):
    raise ValueError("Raw and effective count models use different active-state orderings")

np.random.seed(RANDOM_SEED)
bayesian_msm = BayesianMSM(
    reversible=True,
    n_samples=N_POSTERIOR_SAMPLES,
).fit_fetch(effective_count_model)
msm = bayesian_msm.prior

print(f"Active microstates        : {effective_count_model.state_symbols.tolist()}")
print(f"Selected state fraction  : {effective_count_model.selected_state_fraction:.3f}")
print(f"Selected count fraction  : {effective_count_model.selected_count_fraction:.3f}")
print(f"Raw sliding count total  : {raw_count_model.count_matrix.sum():.0f}")
print(f"Effective count total    : {effective_count_model.count_matrix.sum():.3f}")
print("\nRaw sliding transition-count matrix:")
print(raw_count_model.count_matrix.astype(int))
print("\nEffective transition-count matrix used for inference:")
print(effective_count_model.count_matrix)
print("\nMicrostate transition matrix (maximum-likelihood prior):")
print(msm.transition_matrix)


## 4. Two-state coarse-graining and exact kinetic self-consistency

PCCA+ memberships are obtained once from the base prior. “Open” is assigned to
the coarse state with the larger mean pocket-aperture annotation. The Hummer–
Szabo two-state matrix, populations, MFPT point estimates, posterior intervals,
and corrected network diagram all use these fixed memberships.


In [ ]:
# Pocket aperture is used only to name the two PCCA+ states.
n_wall_resseq = [31, 32, 33, 45, 48, 49, 50, 51]
c_wall_resseq = [100, 101, 102, 106, 107, 109, 127, 128, 129, 130, 131, 132, 133]
n_wall = [r.index for r in traj.topology.residues if r.resSeq in n_wall_resseq]
c_wall = [r.index for r in traj.topology.residues if r.resSeq in c_wall_resseq]
wall_pairs = np.asarray([(i, j) for i in n_wall for j in c_wall], dtype=int)
wall_distances, _ = md.compute_contacts(traj, contacts=wall_pairs, scheme="closest-heavy")
pocket_aperture = wall_distances.mean(axis=1)

pcca2 = msm.pcca(2)
memberships = np.asarray(pcca2.memberships)
crisp_active = np.argmax(memberships, axis=1)
active_symbols = np.asarray(effective_count_model.state_symbols, dtype=int)
micro_to_internal_macro = {
    int(symbol): int(crisp_active[i]) for i, symbol in enumerate(active_symbols)
}
frame_internal_macro = np.asarray([
    micro_to_internal_macro[int(state)] for state in all_dtrajs
], dtype=int)

aperture_by_internal_state = np.asarray([
    pocket_aperture[frame_internal_macro == state].mean() for state in range(2)
])
open_internal = int(np.argmax(aperture_by_internal_state))
closed_internal = 1 - open_internal
published_order = [closed_internal, open_internal]
frame_macro = np.where(frame_internal_macro == closed_internal, 0, 1).astype(int)

def hummer_szabo_coarse_grain(transition_matrix, stationary_distribution, chi):
    diagonal_pi = np.diag(stationary_distribution)
    overlap = chi.T @ diagonal_pi @ chi
    propagated = chi.T @ diagonal_pi @ transition_matrix @ chi
    coarse_transition = np.linalg.solve(overlap, propagated)
    coarse_transition = np.real_if_close(coarse_transition)
    coarse_transition /= coarse_transition.sum(axis=1, keepdims=True)
    coarse_pi = chi.T @ stationary_distribution
    coarse_pi /= coarse_pi.sum()
    return coarse_transition, coarse_pi

T2_internal, pi2_internal = hummer_szabo_coarse_grain(
    msm.transition_matrix,
    msm.stationary_distribution,
    memberships,
)
T2 = T2_internal[np.ix_(published_order, published_order)]
pi2 = pi2_internal[published_order]

lag_ns = MSM_LAG_FRAMES * NS_PER_FRAME
mfpt_point_us = np.asarray([lag_ns / T2[0, 1], lag_ns / T2[1, 0]]) / 1000.0
lambda2_macro = np.sort(np.real(np.linalg.eigvals(T2)))[-2]
macro_relaxation_us = -lag_ns / np.log(lambda2_macro) / 1000.0
micro_relaxation_us = float(msm.timescales()[0] * NS_PER_FRAME / 1000.0)

T2_samples = []
pi2_samples = []
mfpt_samples_us = []
for sample in bayesian_msm.samples:
    Ts_internal, pis_internal = hummer_szabo_coarse_grain(
        sample.transition_matrix,
        sample.stationary_distribution,
        memberships,
    )
    Ts = Ts_internal[np.ix_(published_order, published_order)]
    pis = pis_internal[published_order]
    T2_samples.append(Ts)
    pi2_samples.append(pis)
    mfpt_samples_us.append([lag_ns / Ts[0, 1] / 1000.0, lag_ns / Ts[1, 0] / 1000.0])

T2_samples = np.asarray(T2_samples)
pi2_samples = np.asarray(pi2_samples)
mfpt_samples_us = np.asarray(mfpt_samples_us)

def median_ci(values):
    return np.percentile(np.asarray(values), [50.0, 2.5, 97.5], axis=0)

pi_summary = median_ci(pi2_samples)
mfpt_summary = median_ci(mfpt_samples_us)
T2_summary = median_ci(T2_samples)

print(f"PCCA aperture means (nm), internal order: {aperture_by_internal_state}")
print(f"Published state order: closed={closed_internal}, open={open_internal}")
print("\nTwo-state transition matrix at 50 ns [Closed, Open]:")
print(T2)
print(f"Point stationary populations: Closed {pi2[0]:.3%}, Open {pi2[1]:.3%}")
print(f"Posterior median populations (95% CrI):")
print(f"  Closed {pi_summary[0,0]:.3%} ({pi_summary[1,0]:.3%}–{pi_summary[2,0]:.3%})")
print(f"  Open   {pi_summary[0,1]:.3%} ({pi_summary[1,1]:.3%}–{pi_summary[2,1]:.3%})")
print("\nModel-extrapolated MFPTs:")
print(f"  Closed -> Open point estimate: {mfpt_point_us[0]:.3f} µs")
print(f"    posterior median (95% CrI): {mfpt_summary[0,0]:.3f} "
      f"({mfpt_summary[1,0]:.3f}–{mfpt_summary[2,0]:.3f}) µs")
print(f"  Open -> Closed point estimate: {mfpt_point_us[1]:.3f} µs")
print(f"    posterior median (95% CrI): {mfpt_summary[0,1]:.3f} "
      f"({mfpt_summary[1,1]:.3f}–{mfpt_summary[2,1]:.3f}) µs")
print(f"\nRelaxation time from two-state eigenvalue: {macro_relaxation_us:.3f} µs")
print(f"Slowest microstate implied timescale     : {micro_relaxation_us:.3f} µs")
print(f"MFPT identity check                     : "
      f"{1.0 / (1.0 / mfpt_point_us[0] + 1.0 / mfpt_point_us[1]):.3f} µs")

print("\nExact transition probabilities from matrix powers:")
for k in CK_MULTIPLES:
    Tk = np.linalg.matrix_power(T2, k)
    print(f"  {k * lag_ns:6.1f} ns: Closed->Open={Tk[0,1]:.6f}, "
          f"Open->Closed={Tk[1,0]:.6f}")


In [ ]:
# Corrected Figure 4g: arrows and labels follow the actual direction of travel.
inferno = plt.cm.inferno
state_colors = [inferno(0.35), inferno(0.72)]
positions = [(-0.85, 0.0), (0.85, 0.0)]
radii = 0.34 * np.sqrt(pi2 / pi2.max())

fig, ax = plt.subplots(figsize=(8.4, 4.8))
ax.set_xlim(-1.55, 1.55)
ax.set_ylim(-1.05, 1.05)
ax.set_aspect("equal")
ax.axis("off")

for i, (name, xy) in enumerate(zip(["Closed", "Open"], positions)):
    ax.add_patch(Circle(xy, radii[i], facecolor=state_colors[i], edgecolor="black", lw=1.3, zorder=3))
    ax.text(*xy, f"{name}\n{pi2[i]:.1%}", ha="center", va="center",
            color="white", fontsize=13, fontweight="bold", zorder=4)

ax.add_patch(FancyArrowPatch(
    (positions[0][0] + radii[0], 0.13), (positions[1][0] - radii[1], 0.13),
    connectionstyle="arc3,rad=0.28", arrowstyle="-|>", mutation_scale=18,
    lw=1.7, color="0.25", zorder=2,
))
ax.text(0, 0.62,
        f"Closed → Open\n{mfpt_point_us[0]:.1f} µs "
        f"({mfpt_summary[1,0]:.1f}–{mfpt_summary[2,0]:.1f})",
        ha="center", va="center", fontsize=10.5)

ax.add_patch(FancyArrowPatch(
    (positions[1][0] - radii[1], -0.13), (positions[0][0] + radii[0], -0.13),
    connectionstyle="arc3,rad=0.28", arrowstyle="-|>", mutation_scale=18,
    lw=1.7, color="0.25", zorder=2,
))
ax.text(0, -0.62,
        f"Open → Closed\n{mfpt_point_us[1]:.1f} µs "
        f"({mfpt_summary[1,1]:.1f}–{mfpt_summary[2,1]:.1f})",
        ha="center", va="center", fontsize=10.5)

ax.text(0.5, 0.01,
        "Point estimate (95% Bayesian credible interval); model-extrapolated",
        transform=ax.transAxes, ha="center", fontsize=8.5, color="0.35")
fig.tight_layout()
save_figure(fig, "Figure4g_MSM_network_corrected")
plt.show()


## 5. Implied timescales versus lag

The TICA projection and six microstates are held fixed while the reversible MSM
is refit at each lag. This is the lag-sensitivity diagnostic previously missing
from the supplementary material; it is not a substitute for trajectory-wise
cross-validation or full model-selection sensitivity analysis.


In [ ]:
its_values_us = []
its_connectivity = []
for lag in ITS_LAGS_FRAMES:
    count_at_lag = (
        TransitionCountEstimator(lagtime=lag, count_mode=MSM_COUNT_MODE)
        .fit_fetch(dtrajs)
        .submodel_largest()
    )
    lag_model = MaximumLikelihoodMSM(reversible=True).fit_fetch(count_at_lag)
    ts = np.asarray(lag_model.timescales()) * NS_PER_FRAME / 1000.0
    its_values_us.append(ts[:3])
    its_connectivity.append([
        lag,
        count_at_lag.n_states,
        count_at_lag.selected_state_fraction,
        count_at_lag.selected_count_fraction,
    ])

its_values_us = np.asarray(its_values_us)
lag_ns_values = np.asarray(ITS_LAGS_FRAMES) * NS_PER_FRAME

fig, ax = plt.subplots(figsize=(7.0, 5.6))
for i in range(its_values_us.shape[1]):
    ax.plot(lag_ns_values, its_values_us[:, i], marker="o", ms=4.5, lw=1.5,
            label=f"timescale {i + 1}")
ax.plot(lag_ns_values, lag_ns_values / 1000.0, ls="--", lw=1.0, color="0.45",
        label="timescale = lag")
ax.axvline(MSM_LAG_FRAMES * NS_PER_FRAME, color="#B2182B", ls=":", lw=1.4,
           label=f"selected MSM lag ({MSM_LAG_FRAMES * NS_PER_FRAME:.0f} ns)")
ax.set_yscale("log")
ax.set_xlabel("MSM lag (ns)")
ax.set_ylabel("implied timescale (µs)")
ax.grid(False, which="both")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
save_figure(fig, "Supplementary_implied_timescales_no_grid")
plt.show()

print("lag(frames)  lag(ns)  active/6  state_fraction  count_fraction  t1(us)")
for row, ts in zip(its_connectivity, its_values_us):
    lag, n_active, state_fraction, count_fraction = row
    print(f"{int(lag):10d}  {lag * NS_PER_FRAME:7.1f}  {int(n_active):>3d}/6  "
          f"{state_fraction:14.3f}  {count_fraction:14.3f}  {ts[0]:7.3f}")


## 6. Corrected two-state Chapman–Kolmogorov test

The 50-ns object in `ck_models[0]` is the already fitted `bayesian_msm`; it is
not a separately fitted or stale model. Predictions are explicit powers of the
reported Hummer–Szabo matrix, $T_2(50\,\mathrm{ns})^k$. Direct estimates at each
$k\tau$ are obtained by refitting the microstate MSM and applying the **same
fixed PCCA+ memberships and the same Hummer–Szabo projection**.

This explicit construction is necessary because deeptime's generic
`BayesianMSM.ck_test(..., n_metastable_sets=2)` reports fuzzy-set correlation/
overlap observables. Those values are useful for their own purpose but are not
the closed→open probabilities in the two-state transition matrix and must not
be compared directly with the MFPTs.


In [ ]:
ck_lags_frames = [MSM_LAG_FRAMES * k for k in CK_MULTIPLES]
ck_models = [bayesian_msm]
ck_count_report = [[
    MSM_LAG_FRAMES,
    effective_count_model.n_states,
    effective_count_model.selected_state_fraction,
    effective_count_model.selected_count_fraction,
    float(effective_count_model.count_matrix.sum()),
]]

for lag in ck_lags_frames[1:]:
    count_at_lag = (
        TransitionCountEstimator(lagtime=lag, count_mode=MSM_COUNT_MODE)
        .fit_fetch(dtrajs)
        .submodel_largest()
    )
    np.random.seed(RANDOM_SEED + lag)
    ck_models.append(BayesianMSM(reversible=True, n_samples=300).fit_fetch(count_at_lag))
    ck_count_report.append([
        lag,
        count_at_lag.n_states,
        count_at_lag.selected_state_fraction,
        count_at_lag.selected_count_fraction,
        float(count_at_lag.count_matrix.sum()),
    ])

ck_lags_ns = np.asarray(ck_lags_frames, dtype=float) * NS_PER_FRAME
ck_predictions = np.asarray([
    np.linalg.matrix_power(T2, k) for k in CK_MULTIPLES
])
prediction_samples = np.asarray([
    [np.linalg.matrix_power(sample_T2, k) for sample_T2 in T2_samples]
    for k in CK_MULTIPLES
])

ck_estimates = []
direct_samples_by_lag = []
for lag_model in ck_models:
    direct_internal, _ = hummer_szabo_coarse_grain(
        lag_model.prior.transition_matrix,
        lag_model.prior.stationary_distribution,
        memberships,
    )
    ck_estimates.append(direct_internal[np.ix_(published_order, published_order)])

    samples_at_lag = []
    for sample in lag_model.samples:
        sample_internal, _ = hummer_szabo_coarse_grain(
            sample.transition_matrix,
            sample.stationary_distribution,
            memberships,
        )
        samples_at_lag.append(sample_internal[np.ix_(published_order, published_order)])
    direct_samples_by_lag.append(samples_at_lag)

ck_estimates = np.asarray(ck_estimates)
direct_samples_by_lag = [np.asarray(x) for x in direct_samples_by_lag]
prediction_low = np.percentile(prediction_samples, 2.5, axis=1)
prediction_high = np.percentile(prediction_samples, 97.5, axis=1)
estimate_low = np.asarray([np.percentile(x, 2.5, axis=0) for x in direct_samples_by_lag])
estimate_high = np.asarray([np.percentile(x, 97.5, axis=0) for x in direct_samples_by_lag])

BLUE, RED = "#2166AC", "#B2182B"
labels = ["Closed", "Open"]
fig, axes = plt.subplots(2, 2, figsize=(8.0, 6.8), sharex=True)
for i in range(2):
    for j in range(2):
        ax = axes[i, j]
        ax.fill_between(ck_lags_ns, prediction_low[:, i, j], prediction_high[:, i, j],
                        color=BLUE, alpha=0.16, lw=0)
        ax.plot(ck_lags_ns, ck_predictions[:, i, j], color=BLUE, lw=2.0,
                label="base-MSM prediction")
        ax.fill_between(ck_lags_ns, estimate_low[:, i, j], estimate_high[:, i, j],
                        color=RED, alpha=0.16, lw=0)
        ax.plot(ck_lags_ns, ck_estimates[:, i, j], color=RED, ls="--", lw=1.2)
        ax.scatter(ck_lags_ns, ck_estimates[:, i, j], color=RED, edgecolor="black",
                   linewidth=0.4, s=28, zorder=4, label="direct estimate")
        ax.set_title(f"{labels[i]} → {labels[j]}", fontsize=11)
        ax.set_ylim((0.989, 1.001) if i == j else (-0.0005, 0.0085))
        ax.grid(False, which="both")
        if i == 1:
            ax.set_xlabel("lag time (ns)")
        if j == 0:
            ax.set_ylabel("transition probability")
handles, legend_labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, legend_labels, loc="upper center", ncol=2, frameon=False)
fig.suptitle("Explicit same-observable two-state CK test (expanded y-axes)", y=0.96, fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.91])
save_figure(fig, "Figure4h_CK_2state_corrected_no_grid")
plt.show()

ck_max_abs_deviation = float(np.max(np.abs(ck_predictions - ck_estimates)))
same_base_max_abs_difference = float(np.max(np.abs(ck_predictions[0] - T2)))
offdiag_prediction = np.stack([ck_predictions[:, 0, 1], ck_predictions[:, 1, 0]], axis=1)
offdiag_estimate = np.stack([ck_estimates[:, 0, 1], ck_estimates[:, 1, 0]], axis=1)
offdiag_ratio = np.divide(
    offdiag_prediction,
    offdiag_estimate,
    out=np.full_like(offdiag_prediction, np.nan),
    where=offdiag_estimate > 0,
)
print("lag(frames)  lag(ns)  active/6  state_fraction  count_fraction  effective_counts")
for lag, n_active, state_fraction, count_fraction, total in ck_count_report:
    print(f"{int(lag):10d}  {lag * NS_PER_FRAME:7.1f}  {int(n_active):>3d}/6  "
          f"{state_fraction:14.3f}  {count_fraction:14.3f}  {total:16.3f}")
print(f"\nMaximum |prediction - direct estimate|: {ck_max_abs_deviation:.6f}")
print(f"Maximum |CK base matrix - reported T2| : {same_base_max_abs_difference:.3e}")
print(f"Prediction/direct off-diagonal ratios [C->O, O->C]:\n{offdiag_ratio}")
print("\nCK numerical values:")
for n, lag_ns_i in enumerate(ck_lags_ns):
    print(f"  {lag_ns_i:6.1f} ns  predicted C->O={ck_predictions[n,0,1]:.6f}, "
          f"direct C->O={ck_estimates[n,0,1]:.6f}; "
          f"predicted O->C={ck_predictions[n,1,0]:.6f}, "
          f"direct O->C={ck_estimates[n,1,0]:.6f}")


## 7. Replica-resolved macrostate timelines and observed counts

The one-frame counts below are literal changes in the hard PCCA+ assignment at
1-ns resolution. The 50-frame counts are overlapping lagged pairs, not numbers
of statistically independent transition events. Both are descriptive and must
not be presented as dozens of independent barrier crossings.


In [ ]:
def two_state_count_matrix(sequence, lag):
    sequence = np.asarray(sequence, dtype=int)
    out = np.zeros((2, 2), dtype=int)
    if len(sequence) <= lag:
        return out
    np.add.at(out, (sequence[:-lag], sequence[lag:]), 1)
    return out

replica_macro_trajectories = [frame_macro[s:e] for s, e in rep_boundaries]
transition_count_rows = []

fig, axes = plt.subplots(4, 1, figsize=(10.5, 7.4), sharey=True)
for rep_i, (ax, sequence) in enumerate(zip(axes, replica_macro_trajectories), start=1):
    time_us = np.arange(len(sequence)) * NS_PER_FRAME / 1000.0
    ax.step(time_us, sequence, where="post", lw=0.75, color="#2166AC")
    ax.set_ylim(-0.18, 1.18)
    ax.set_yticks([0, 1], ["Closed", "Open"])
    ax.set_ylabel(f"Rep {rep_i}")
    ax.grid(axis="x", alpha=0.22)
    c1 = two_state_count_matrix(sequence, 1)
    c50 = two_state_count_matrix(sequence, MSM_LAG_FRAMES)
    occupancy = np.bincount(sequence, minlength=2) / len(sequence)
    transition_count_rows.append({
        "replica": rep_i,
        "frames": len(sequence),
        "closed_occupancy": occupancy[0],
        "open_occupancy": occupancy[1],
        "one_frame_closed_to_open": int(c1[0, 1]),
        "one_frame_open_to_closed": int(c1[1, 0]),
        "lag50_closed_to_open_pairs": int(c50[0, 1]),
        "lag50_open_to_closed_pairs": int(c50[1, 0]),
    })
axes[-1].set_xlabel("time within replica (µs)")
fig.suptitle("Raw hard-assignment macrostate timelines (1 ns/frame)", y=0.995)
fig.tight_layout()
save_figure(fig, "Supplementary_macrostate_timelines")
plt.show()

print("rep  frames  closed_occ  open_occ  1-frame C->O/O->C  50-frame C->O/O->C")
for row in transition_count_rows:
    print(f"{row['replica']:3d}  {row['frames']:6d}  "
          f"{row['closed_occupancy']:10.3f}  {row['open_occupancy']:8.3f}  "
          f"{row['one_frame_closed_to_open']:7d}/{row['one_frame_open_to_closed']:<7d}  "
          f"{row['lag50_closed_to_open_pairs']:8d}/{row['lag50_open_to_closed_pairs']:<8d}")

aggregate_c1 = sum((two_state_count_matrix(x, 1) for x in replica_macro_trajectories),
                   start=np.zeros((2, 2), dtype=int))
aggregate_c50 = sum((two_state_count_matrix(x, MSM_LAG_FRAMES) for x in replica_macro_trajectories),
                    start=np.zeros((2, 2), dtype=int))
print("\nAggregate one-frame hard-macrostate count matrix:")
print(aggregate_c1)
print("Aggregate 50-frame sliding hard-macrostate pair matrix:")
print(aggregate_c50)


## 8. Provenance dump and serialized numeric MSM bundle

The NPZ bundle contains the exact matrices and assignments used above. CSV/JSON
companions make the audit readable without loading a Python object whose format
may depend on a particular deeptime version.


In [ ]:
eigenvalues = np.real(np.asarray(msm.eigenvalues()))
implied_timescales_ns = np.real(np.asarray(msm.timescales())) * NS_PER_FRAME

print(f"frame spacing              : {NS_PER_FRAME} ns/frame")
print(f"TICA lag                   : {TICA_LAG_FRAMES} frames = "
      f"{TICA_LAG_FRAMES * NS_PER_FRAME} ns")
print(f"MSM lag                    : {MSM_LAG_FRAMES} frames = "
      f"{MSM_LAG_FRAMES * NS_PER_FRAME} ns")
print(f"MSM count mode             : {MSM_COUNT_MODE}")
print(f"Bayesian posterior samples : {N_POSTERIOR_SAMPLES}")
print(f"Active set                 : {active_symbols.tolist()}")
print(f"Eigenvalues                : {eigenvalues}")
print(f"Implied timescales (ns)    : {implied_timescales_ns}")
print(f"Implied timescales (µs)    : {implied_timescales_ns / 1000.0}")
print("\nExact time conversions used:")
print("  lag_ns = lag_frames * NS_PER_FRAME")
print("  timescale_us = timescale_frames * NS_PER_FRAME / 1000")
print("  two_state_MFPT_us = lag_frames * NS_PER_FRAME / T_ij / 1000")
print("  k-lag CK prediction = matrix_power(T2, k)")

np.savez_compressed(
    RESULTS_DIR / "apo16e6_msm_numeric_bundle.npz",
    frame_spacing_ns=np.asarray(NS_PER_FRAME),
    tica_lag_frames=np.asarray(TICA_LAG_FRAMES),
    msm_lag_frames=np.asarray(MSM_LAG_FRAMES),
    replica_lengths=np.asarray(REPLICA_LENGTHS),
    feature_pairs_residue_indices=feature_pairs,
    discrete_trajectories_concatenated=all_dtrajs,
    replica_boundaries=np.asarray(rep_boundaries),
    active_microstates=active_symbols,
    raw_sliding_count_matrix=raw_count_model.count_matrix,
    effective_count_matrix=effective_count_model.count_matrix,
    micro_transition_matrix=msm.transition_matrix,
    micro_stationary_distribution=msm.stationary_distribution,
    micro_eigenvalues=eigenvalues,
    micro_implied_timescales_ns=implied_timescales_ns,
    pcca_memberships=memberships,
    hard_macrostate_trajectory_closed0_open1=frame_macro,
    coarse_transition_matrix_closed_open=T2,
    coarse_stationary_distribution_closed_open=pi2,
    posterior_coarse_transition_matrices=T2_samples,
    posterior_coarse_stationary_distributions=pi2_samples,
    posterior_mfpt_us=mfpt_samples_us,
    ck_lags_ns=ck_lags_ns,
    ck_predictions=ck_predictions,
    ck_direct_estimates=ck_estimates,
    ck_prediction_to_direct_offdiagonal_ratio=offdiag_ratio,
)

np.savetxt(RESULTS_DIR / "micro_count_matrix_raw_sliding.csv",
           raw_count_model.count_matrix, delimiter=",", fmt="%.0f")
np.savetxt(RESULTS_DIR / "micro_count_matrix_effective.csv",
           effective_count_model.count_matrix, delimiter=",", fmt="%.10g")
np.savetxt(RESULTS_DIR / "micro_transition_matrix.csv",
           msm.transition_matrix, delimiter=",", fmt="%.12g")
np.savetxt(RESULTS_DIR / "coarse_transition_matrix_closed_open.csv",
           T2, delimiter=",", fmt="%.12g")
np.savetxt(RESULTS_DIR / "feature_pairs_resseq.csv",
           np.asarray([(resseq_by_index[i], resseq_by_index[j]) for i, j in feature_pairs]),
           delimiter=",", fmt="%d", header="resSeq_i,resSeq_j", comments="")

with (RESULTS_DIR / "replica_macrostate_transition_counts.csv").open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(transition_count_rows[0]))
    writer.writeheader()
    writer.writerows(transition_count_rows)

with (RESULTS_DIR / "ck_numerical_values.csv").open("w", newline="") as handle:
    writer = csv.writer(handle)
    writer.writerow([
        "lag_ns", "pred_closed_closed", "pred_closed_open",
        "pred_open_closed", "pred_open_open", "direct_closed_closed",
        "direct_closed_open", "direct_open_closed", "direct_open_open",
    ])
    for lag_ns_i, prediction, estimate in zip(ck_lags_ns, ck_predictions, ck_estimates):
        writer.writerow([lag_ns_i, *prediction.ravel().tolist(), *estimate.ravel().tolist()])

metadata = {
    "software": {
        "python": platform.python_version(),
        "deeptime": deeptime.__version__,
        "mdtraj": md.__version__,
        "numpy": np.__version__,
        "matplotlib": matplotlib.__version__,
    },
    "random_seed": RANDOM_SEED,
    "frame_spacing_ns": NS_PER_FRAME,
    "timebase_status": timebase_status,
    "xtc_detected_spacing_ns_by_replica": detected_ns_per_frame_by_replica,
    "xtc_timestamp_span_us_by_replica": timestamp_duration_us_by_replica,
    "replica_lengths_frames": list(REPLICA_LENGTHS),
    "feature_definition": "closest-heavy distances for all nonadjacent residue pairs in resSeq 62-79",
    "n_features": int(len(feature_pairs)),
    "tica_lag_frames": TICA_LAG_FRAMES,
    "tica_dim": TICA_DIM,
    "n_microstates": N_MICROSTATES,
    "kmeans_seed": RANDOM_SEED,
    "msm_lag_frames": MSM_LAG_FRAMES,
    "msm_count_mode": MSM_COUNT_MODE,
    "reversible": True,
    "bayesian_n_samples": N_POSTERIOR_SAMPLES,
    "bayesian_n_steps": "deeptime automatic default (None)",
    "coarse_graining": "PCCA+ memberships fixed from the maximum-likelihood prior; Hummer-Szabo projection",
    "ck_definition": "powers of the reported 50-frame Hummer-Szabo matrix versus direct Hummer-Szabo projection of MSMs refit at each lag, using fixed base-model PCCA+ memberships",
    "state_order": ["Closed", "Open"],
    "state_label_annotation": "mean closest-heavy pocket aperture; larger mean is Open",
    "mfpt_reporting": "maximum-likelihood point estimate plus posterior median and 2.5/97.5 percentiles",
    "ck_lags_frames": ck_lags_frames,
    "limitations": [
        "all replicas started from the same bound-like structure",
        "CK lags are much shorter than the slowest implied timescale",
        "posterior holds TICA, clustering, and PCCA memberships fixed",
        "hard-assignment lag-pair counts are overlapping and not independent transition events",
    ],
}
with (RESULTS_DIR / "apo16e6_msm_metadata.json").open("w") as handle:
    json.dump(metadata, handle, indent=2)

print(f"\nSaved audit bundle and tables to: {RESULTS_DIR.resolve()}")
for path in sorted(RESULTS_DIR.iterdir()):
    print(f"  {path.name}")


## Interpretation boundary

The prior CK panel and the reported MFPTs represented different coarse-grained
quantities. The explicit same-matrix test above resolves that semantic mismatch,
but the current model still does **not** pass a persuasive kinetic validation:
the slowest implied timescale rises sharply beyond the selected 50-ns lag, and
the direct Hummer–Szabo off-diagonal estimates at 100–250 ns are much smaller
than the powers of the 50-ns matrix. A small absolute CK deviation is misleading
when the probabilities themselves are only $10^{-3}$–$10^{-2}$; relative
differences and credible intervals must also be examined.

It remains scientifically inappropriate to treat the two-state populations and
MFPTs as precisely demonstrated equilibrium quantities: all replicas began from
one bound-like structure, individual trajectories are shorter than the slowest
implied timescale, and uncertainty sampling here holds the learned feature
projection, clustering, and PCCA+ state definition fixed. Until stronger
sampling and model-selection validation are completed, manuscript language
should describe observed conformational heterogeneity and model-extrapolated
kinetics rather than established equilibrium populations or “tens of
microseconds” interconversion.
